# Mahie Bulk T-Testing

This notebook performs bulk event-locked LFP statistical testing from precomputed collection HDF5 files. It is organized into annotated sections so the analysis flow is easier to review now and easier to migrate into `src/coop_ephys_analysis/lfp/` later.


## Imports

Import standard-library utilities, data containers, numerical/scientific packages, HDF5 support, and statistical functions.


In [ ]:
from __future__ import annotations

import math
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import h5py
import numpy as np
import pandas as pd
from scipy import stats


warnings.filterwarnings("ignore", category=RuntimeWarning)


## Paths

Locate the LFP collection inputs and define where statistical tables and reports will be written.


In [ ]:
DOWNLOADS = Path.home() / "Downloads"

OUTPUT_DIR = (
    DOWNLOADS
    / "xgboost_feature_finding"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def find_collection_folder(
    folder_name: str,
) -> Path:
    """
    Find a named collection folder anywhere under Downloads.

    The direct Downloads/folder_name location is checked first.
    """

    direct_path = (
        DOWNLOADS
        / folder_name
    )

    if direct_path.is_dir():
        return direct_path.resolve()

    matches = sorted(
        path.resolve()
        for path in DOWNLOADS.rglob(
            folder_name
        )
        if path.is_dir()
    )

    if not matches:
        raise FileNotFoundError(
            f"Could not find a folder named "
            f"{folder_name!r} anywhere under "
            f"{DOWNLOADS}."
        )

    if len(matches) > 1:
        formatted_matches = "\n".join(
            f"  - {path}"
            for path in matches
        )

        raise RuntimeError(
            f"Found more than one "
            f"{folder_name!r} folder.\n"
            "Keep only the intended collection "
            "under Downloads or manually set "
            "COLLECTION_PATHS.\n"
            f"{formatted_matches}"
        )

    return matches[0]


COLLECTION_PATHS = {
    "day 1": find_collection_folder(
        "D1_lfp_collection"
    ),
    "day 8": find_collection_folder(
        "D8_lfp_collection"
    ),
}


## Analysis Settings

Set the statistical threshold, event windows, canonical time step, nose-poke filtering rules, Granger transform, and frequency bands.


In [ ]:
ALPHA = 0.05

PRE_SECONDS = 2.0
POST_SECONDS = 2.0

# Common time grid used when combining recordings.
CANONICAL_TIME_STEP = 0.5

EPSILON = 1e-12

NOSE_POKE_LOCKOUT_SECONDS = 5.0

REQUIRE_REWARD_IF_REWARD_EVENTS_EXIST = True

REWARD_MATCH_MIN_SECONDS = 0.0
REWARD_MATCH_MAX_SECONDS = 4.0

GRANGER_TRANSFORM = "log1p"


# Lower boundaries are inclusive.
# Upper boundaries are exclusive except for the
# final 100-Hz gamma boundary.

FREQUENCY_BANDS = (
    (
        "delta",
        0.5,
        4.0,
    ),
    (
        "theta",
        4.0,
        8.0,
    ),
    (
        "alpha",
        8.0,
        12.0,
    ),
    (
        "beta",
        12.0,
        30.0,
    ),
    (
        "gamma",
        30.0,
        100.0,
    ),
)


## Embedded Permitted Metric Rows

Define the exact power regions, coherence edges, and directed Granger edges that enter the bulk statistical tests.


In [ ]:
# These are the exact permitted power regions.

POWER_FEATURES = (
    "mPFC",
    "IC",
    "NAc",
    "MD",
    "LH",
    "BLA",
)


# Nondirectional coherence connections.
#
# Tuple:
#     display label,
#     first region,
#     second region

COHERENCE_FEATURES = (
    (
        "mPFC–BLA",
        "mPFC",
        "BLA",
    ),
    (
        "mPFC–IC",
        "mPFC",
        "IC",
    ),
    (
        "mPFC–MD",
        "mPFC",
        "MD",
    ),
    (
        "mPFC–NAc",
        "mPFC",
        "NAc",
    ),
)


# Directed Granger connections.
#
# Tuple:
#     display label,
#     target region,
#     source region
#
# HDF5 orientation:
#
#     granger[
#         ...,
#         target,
#         source
#     ]
#
# means:
#
#     source → target

GRANGER_FEATURES = (
    (
        "BLA → mPFC",
        "mPFC",
        "BLA",
    ),
    (
        "IC → mPFC",
        "mPFC",
        "IC",
    ),
    (
        "MD → mPFC",
        "mPFC",
        "MD",
    ),
    (
        "NAc → mPFC",
        "mPFC",
        "NAc",
    ),
    (
        "mPFC → BLA",
        "BLA",
        "mPFC",
    ),
    (
        "mPFC → IC",
        "IC",
        "mPFC",
    ),
    (
        "mPFC → MD",
        "MD",
        "mPFC",
    ),
    (
        "mPFC → NAc",
        "NAc",
        "mPFC",
    ),
)


ANALYSIS_REGIONS = tuple(
    dict.fromkeys(
        list(
            POWER_FEATURES
        )
        + [
            region
            for (
                _,
                first_region,
                second_region,
            ) in COHERENCE_FEATURES
            for region in (
                first_region,
                second_region,
            )
        ]
        + [
            region
            for (
                _,
                target_region,
                source_region,
            ) in GRANGER_FEATURES
            for region in (
                target_region,
                source_region,
            )
        ]
    )
)


METRICS = (
    "power",
    "coherence",
    "granger",
)


BEHAVIORS = (
    "light",
    "nose poke",
)


## Animal Roles

Assign animal IDs to subject, recipient, and subject-alone roles so recordings can be mapped into behavioral conditions.


In [ ]:
# These assignments are taken directly from the
# original spectrogram notebook.

SUBJECT_IDS = {
    "1-2",
    "2-4",
    "4-4",
    "4-3",
}


RECIPIENT_IDS = {
    "1-3",
    "2-1",
    "4-2",
    "4-1",
}


ALONE_IDS = {
    "6-1",
    "6-3",
}


## Conditions

Define comparison groups and condition metadata used by the downstream t-tests.


In [ ]:
CONDITIONS = {
    "selfish": {
        "role": "subject",
        "choice": "selfish",
        "display": "selfish",
    },
    "coop": {
        "role": "subject",
        "choice": "prosocial",
        "display": "coop",
    },
    "selfish_alone": {
        "role": "subject_alone",
        "choice": "selfish",
        "display": "selfish alone",
    },
    "coop_alone": {
        "role": "subject_alone",
        "choice": "prosocial",
        "display": "coop alone",
    },
}


@dataclass(
    frozen=True
)
class Comparison:
    key: str
    filename: str
    display: str
    group_1: str
    group_2: str
    paired: bool


COMPARISONS = (
    Comparison(
        key=(
            "selfish_alone_vs_"
            "coop_alone"
        ),
        filename=(
            "01_selfish_alone_"
            "vs_coop_alone.csv"
        ),
        display=(
            "selfish alone vs. "
            "coop alone"
        ),
        group_1="selfish_alone",
        group_2="coop_alone",
        paired=True,
    ),
    Comparison(
        key=(
            "selfish_vs_"
            "selfish_alone"
        ),
        filename=(
            "02_selfish_vs_"
            "selfish_alone.csv"
        ),
        display=(
            "selfish vs. "
            "selfish alone"
        ),
        group_1="selfish",
        group_2="selfish_alone",
        paired=False,
    ),
    Comparison(
        key=(
            "coop_vs_"
            "coop_alone"
        ),
        filename=(
            "03_coop_vs_"
            "coop_alone.csv"
        ),
        display=(
            "coop vs. "
            "coop alone"
        ),
        group_1="coop",
        group_2="coop_alone",
        paired=False,
    ),
    Comparison(
        key="selfish_vs_coop",
        filename=(
            "04_selfish_vs_"
            "coop.csv"
        ),
        display=(
            "selfish vs. coop"
        ),
        group_1="selfish",
        group_2="coop",
        paired=True,
    ),
)


@dataclass
class Recording:
    day: str
    name: str
    path: Path
    subject: Any
    animal_id: str
    role: str
    time_step: float
    window_duration: float
    frequencies: np.ndarray
    region_to_index: dict[str, int]
    events: dict[str, np.ndarray]


## Basic Helpers

Provide small utilities for HDF5 decoding, numeric conversion, animal ID parsing, role lookup, resampling, and display formatting.


In [ ]:
def decode_h5_value(
    value: Any,
) -> Any:
    """Decode HDF5 byte strings and NumPy scalars."""

    if isinstance(
        value,
        bytes,
    ):
        return value.decode(
            "utf-8"
        )

    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    return value


def safe_float(
    value: Any,
    default: float,
) -> float:
    """Convert a metadata value to a finite float."""

    try:
        result = float(
            decode_h5_value(
                value
            )
        )

    except (
        TypeError,
        ValueError,
    ):
        return default

    if not math.isfinite(
        result
    ):
        return default

    return result


def extract_animal_id(
    recording_name: str,
    subject: Any = None,
) -> str | None:
    """Extract an animal ID such as 1-2."""

    for token in str(
        recording_name
    ).split("_"):

        match = re.fullmatch(
            r"(\d+)[.-](\d+)",
            token,
        )

        if match:
            return (
                f"{match.group(1)}-"
                f"{match.group(2)}"
            )

    subject_match = re.fullmatch(
        r"(\d+)[.-](\d+)",
        str(subject),
    )

    if subject_match:
        return (
            f"{subject_match.group(1)}-"
            f"{subject_match.group(2)}"
        )

    return None


def determine_role(
    animal_id: str | None,
) -> str:
    """Assign subject, recipient, alone, or unknown."""

    if animal_id in SUBJECT_IDS:
        return "subject"

    if animal_id in RECIPIENT_IDS:
        return "recipient"

    if animal_id in ALONE_IDS:
        return "subject_alone"

    return "unknown"


def read_integer_mapping(
    group: h5py.Group | None,
) -> dict[str, int]:
    """
    Read a mapping stored as HDF5 attributes
    or scalar datasets.
    """

    mapping: dict[
        str,
        int,
    ] = {}

    if group is None:
        return mapping

    for (
        key,
        value,
    ) in group.attrs.items():

        try:
            mapping[
                str(key)
            ] = int(
                decode_h5_value(
                    value
                )
            )

        except (
            TypeError,
            ValueError,
        ):
            pass

    for (
        key,
        obj,
    ) in group.items():

        if not isinstance(
            obj,
            h5py.Dataset,
        ):
            continue

        try:
            mapping[
                str(key)
            ] = int(
                decode_h5_value(
                    obj[()]
                )
            )

        except (
            TypeError,
            ValueError,
        ):
            pass

    return mapping


def read_events(
    h5_file: h5py.File,
) -> dict[str, np.ndarray]:
    """Read event datasets and event attributes."""

    event_group = h5_file.get(
        "event"
    )

    if event_group is None:
        event_group = h5_file.get(
            "events"
        )

    if event_group is None:
        return {}

    events: dict[
        str,
        np.ndarray,
    ] = {}

    for (
        name,
        obj,
    ) in event_group.items():

        if isinstance(
            obj,
            h5py.Dataset,
        ):
            events[
                str(name)
            ] = np.asarray(
                obj[()]
            )

    for (
        name,
        value,
    ) in event_group.attrs.items():

        events.setdefault(
            str(name),
            np.atleast_1d(
                decode_h5_value(
                    value
                )
            ),
        )

    return events


def nearest_resample_axis(
    array: np.ndarray,
    old_axis: np.ndarray,
    new_axis: np.ndarray,
    axis: int,
) -> np.ndarray:
    """Nearest-neighbor resampling along one axis."""

    old_axis = np.asarray(
        old_axis,
        dtype=float,
    )

    new_axis = np.asarray(
        new_axis,
        dtype=float,
    )

    if (
        old_axis.ndim != 1
        or new_axis.ndim != 1
        or old_axis.size == 0
    ):
        raise ValueError(
            "Resampling axes must be "
            "nonempty one-dimensional arrays."
        )

    if not np.all(
        np.diff(
            old_axis
        ) > 0
    ):
        raise ValueError(
            "The source resampling axis "
            "must be strictly increasing."
        )

    indices = np.array(
        [
            int(
                np.argmin(
                    np.abs(
                        old_axis
                        - value
                    )
                )
            )
            for value in new_axis
        ],
        dtype=int,
    )

    return np.take(
        array,
        indices,
        axis=axis,
    )


def canonical_times() -> np.ndarray:
    """Return -2 through +2 seconds in 0.5-second steps."""

    return np.arange(
        -PRE_SECONDS,
        (
            POST_SECONDS
            + CANONICAL_TIME_STEP
            / 2.0
        ),
        CANONICAL_TIME_STEP,
    )


def format_time_slot(
    time_s: float,
    behavior: str,
) -> str:
    """Return a readable event-relative time label."""

    event_label = (
        "nose poke"
        if behavior == "nose poke"
        else "light"
    )

    if abs(
        time_s
    ) < 1e-12:
        return (
            f"0.0 s "
            f"({event_label})"
        )

    if time_s < 0:
        return (
            f"{abs(time_s):.1f} s "
            f"before {event_label}"
        )

    return (
        f"{time_s:.1f} s "
        f"after {event_label}"
    )


def value_scale_description(
    metric: str,
) -> str:
    """Describe the values entering each t-test."""

    descriptions = {
        "power": (
            "within-recording robust "
            "z-score of log10 power"
        ),
        "coherence": (
            "within-recording robust "
            "z-score of coherence"
        ),
        "granger": (
            "within-recording robust "
            "z-score of log1p Granger value"
        ),
    }

    return descriptions[
        metric
    ]


## Load Collection Metadata

Read recording metadata, region mappings, frequencies, and events from each LFP collection without loading all metric arrays at once.


In [ ]:
def load_collection(
    day: str,
    folder: Path,
) -> tuple[
    list[Recording],
    np.ndarray,
]:
    """Read recording metadata and the common frequency grid."""

    h5_files = sorted(
        folder.rglob(
            "*.h5"
        )
    )

    if not h5_files:
        raise FileNotFoundError(
            f"No .h5 files were found "
            f"inside {folder}."
        )

    recordings: list[
        Recording
    ] = []

    reference_frequencies: (
        np.ndarray
        | None
    ) = None

    for path in h5_files:
        with h5py.File(
            path,
            "r",
        ) as h5_file:

            if "data" not in h5_file:
                raise KeyError(
                    f"{path.name} is missing "
                    "its data group."
                )

            if (
                "frequencies"
                not in h5_file[
                    "data"
                ]
            ):
                raise KeyError(
                    f"{path.name} is missing "
                    "data/frequencies."
                )

            if (
                "metadata"
                in h5_file
            ):
                metadata = dict(
                    h5_file[
                        "metadata"
                    ].attrs
                )

            else:
                metadata = {}

            recording_name = str(
                decode_h5_value(
                    metadata.get(
                        "name",
                        path.stem,
                    )
                )
            )

            subject = decode_h5_value(
                metadata.get(
                    "subject",
                    None,
                )
            )

            time_step = safe_float(
                metadata.get(
                    "window step",
                    metadata.get(
                        "window_step",
                        0.5,
                    ),
                ),
                0.5,
            )

            window_duration = safe_float(
                metadata.get(
                    "window duration",
                    metadata.get(
                        "window_duration",
                        1.0,
                    ),
                ),
                1.0,
            )

            if (
                time_step <= 0
                or window_duration <= 0
            ):
                raise ValueError(
                    "Invalid spectral-window "
                    f"metadata in {path.name}: "
                    f"time step={time_step}, "
                    f"duration={window_duration}."
                )

            frequencies = np.asarray(
                h5_file[
                    "data/frequencies"
                ][()]
            ).squeeze().astype(
                float
            )

            if (
                frequencies.ndim != 1
                or frequencies.size == 0
            ):
                raise ValueError(
                    f"{path.name} has an "
                    "invalid frequency array."
                )

            if not np.all(
                np.isfinite(
                    frequencies
                )
            ):
                raise ValueError(
                    f"{path.name} contains "
                    "nonfinite frequencies."
                )

            if not np.all(
                np.diff(
                    frequencies
                ) > 0
            ):
                raise ValueError(
                    f"{path.name} frequencies "
                    "must be strictly increasing."
                )

            region_to_index = (
                read_integer_mapping(
                    h5_file.get(
                        "brain region dict"
                    )
                )
            )

            missing_regions = [
                region
                for region
                in ANALYSIS_REGIONS
                if region
                not in region_to_index
            ]

            if missing_regions:
                raise ValueError(
                    f"{path.name} is missing "
                    "required brain regions "
                    f"{missing_regions}.\n"
                    "Available mappings: "
                    f"{sorted(region_to_index)}"
                )

            events = read_events(
                h5_file
            )

        animal_id = extract_animal_id(
            recording_name,
            subject,
        )

        role = determine_role(
            animal_id
        )

        recordings.append(
            Recording(
                day=day,
                name=recording_name,
                path=path,
                subject=subject,
                animal_id=(
                    animal_id
                    or "unknown"
                ),
                role=role,
                time_step=time_step,
                window_duration=(
                    window_duration
                ),
                frequencies=frequencies,
                region_to_index=(
                    region_to_index
                ),
                events=events,
            )
        )

        selected_frequencies = (
            frequencies[
                (
                    frequencies
                    >= 0.0
                )
                & (
                    frequencies
                    <= 100.0
                )
            ]
        )

        if (
            selected_frequencies.size
            == 0
        ):
            raise ValueError(
                f"{path.name} contains no "
                "frequencies from 0 to 100 Hz."
            )

        if (
            reference_frequencies
            is None
        ):
            reference_frequencies = (
                selected_frequencies.copy()
            )

    assert (
        reference_frequencies
        is not None
    )

    return (
        recordings,
        reference_frequencies,
    )


## Event Identification

Parse event names and convert stored event arrays into onset times in seconds.


In [ ]:
def event_choice(
    event_name: str,
) -> str | None:
    """Identify selfish or prosocial/coop events."""

    lower_name = str(
        event_name
    ).lower()

    if "selfish" in lower_name:
        return "selfish"

    if any(
        word in lower_name
        for word in (
            "coop",
            "cooperative",
            "prosocial",
        )
    ):
        return "prosocial"

    return None


def event_behavior(
    event_name: str,
) -> str | None:
    """Identify light, nose-poke, or reward events."""

    cleaned_name = re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            event_name
        ).lower(),
    )

    if "nosepoke" in cleaned_name:
        return "nose poke"

    if "reward" in cleaned_name:
        return "reward"

    if "light" in cleaned_name:
        return "light"

    return None


def find_event(
    recording: Recording,
    choice: str,
    behavior: str,
) -> str | None:
    """Find the event dataset for one choice and behavior."""

    matching_events = [
        event_name
        for event_name
        in recording.events
        if (
            event_choice(
                event_name
            ) == choice
            and event_behavior(
                event_name
            ) == behavior
        )
    ]

    if not matching_events:
        return None

    return sorted(
        matching_events,
        key=lambda value: (
            len(value),
            value,
        ),
    )[0]


def event_onsets_seconds(
    recording: Recording,
    event_name: str | None,
    number_of_time_bins: int,
) -> np.ndarray:
    """
    Extract event onsets and convert milliseconds
    to seconds when the stored scale is too large.
    """

    if event_name is None:
        return np.array(
            [],
            dtype=float,
        )

    values = np.asarray(
        recording.events[
            event_name
        ],
        dtype=float,
    )

    if values.size == 0:
        return np.array(
            [],
            dtype=float,
        )

    if values.ndim == 0:
        onsets = np.atleast_1d(
            values
        )

    elif values.ndim == 1:
        onsets = values

    elif (
        values.ndim == 2
        and values.shape[1] == 2
    ):
        onsets = values[
            :,
            0,
        ]

    elif (
        values.ndim == 2
        and values.shape[0] == 2
    ):
        onsets = values[
            0,
            :,
        ]

    elif values.ndim == 2:
        onsets = values[
            :,
            0,
        ]

    else:
        return np.array(
            [],
            dtype=float,
        )

    onsets = np.asarray(
        onsets,
        dtype=float,
    )

    onsets = onsets[
        np.isfinite(
            onsets
        )
    ]

    approximate_duration = (
        number_of_time_bins
        * recording.time_step
    )

    if (
        onsets.size
        and np.nanmax(
            onsets
        ) > (
            1.5
            * approximate_duration
        )
    ):
        onsets = (
            onsets
            / 1000.0
        )

    return np.sort(
        onsets
    )


## Nose-poke Filtering

Apply reward matching and lockout filtering to keep only the intended nose-poke events for subject and alone recordings.


In [ ]:
def reward_matched_nose_pokes(
    nose_pokes: np.ndarray,
    rewards: np.ndarray,
) -> np.ndarray:
    """Retain nose pokes followed by a matching reward."""

    if rewards.size == 0:
        return nose_pokes

    retained: list[
        float
    ] = []

    for nose_poke_time in nose_pokes:
        delays = (
            rewards
            - nose_poke_time
        )

        has_matching_reward = np.any(
            (
                delays
                >= REWARD_MATCH_MIN_SECONDS
            )
            & (
                delays
                <= REWARD_MATCH_MAX_SECONDS
            )
        )

        if has_matching_reward:
            retained.append(
                float(
                    nose_poke_time
                )
            )

    return np.asarray(
        retained,
        dtype=float,
    )


def filtered_nose_pokes(
    recording: Recording,
    number_of_time_bins: int,
) -> dict[
    str,
    np.ndarray,
]:
    """
    Reward-match nose pokes when reward events exist,
    then apply one global five-second lockout across
    selfish and prosocial nose pokes.
    """

    candidates: list[
        tuple[
            float,
            str,
        ]
    ] = []

    for choice in (
        "selfish",
        "prosocial",
    ):
        nose_poke_event = find_event(
            recording,
            choice,
            "nose poke",
        )

        if nose_poke_event is None:
            continue

        nose_poke_onsets = (
            event_onsets_seconds(
                recording,
                nose_poke_event,
                number_of_time_bins,
            )
        )

        reward_event = find_event(
            recording,
            choice,
            "reward",
        )

        reward_onsets = (
            event_onsets_seconds(
                recording,
                reward_event,
                number_of_time_bins,
            )
        )

        if (
            REQUIRE_REWARD_IF_REWARD_EVENTS_EXIST
            and reward_event is not None
            and reward_onsets.size > 0
        ):
            retained_onsets = (
                reward_matched_nose_pokes(
                    nose_poke_onsets,
                    reward_onsets,
                )
            )

        else:
            retained_onsets = (
                nose_poke_onsets
            )

        candidates.extend(
            (
                float(onset),
                nose_poke_event,
            )
            for onset
            in retained_onsets
        )

    candidates.sort(
        key=lambda item: item[0]
    )

    retained_by_event: dict[
        str,
        list[float],
    ] = {}

    previous_retained_onset = (
        -np.inf
    )

    for (
        onset,
        event_name,
    ) in candidates:

        if (
            onset
            - previous_retained_onset
            >= NOSE_POKE_LOCKOUT_SECONDS
        ):
            retained_by_event.setdefault(
                event_name,
                [],
            ).append(
                onset
            )

            previous_retained_onset = (
                onset
            )

    return {
        event_name: np.asarray(
            onsets,
            dtype=float,
        )
        for (
            event_name,
            onsets,
        ) in retained_by_event.items()
    }


## Metric Loading

Load one requested LFP metric from HDF5 and restrict it to the analysis frequency and region axes.


In [ ]:
def metric_dataset_name(
    data_group: h5py.Group,
    metric: str,
) -> str | None:
    """Support both granger and older grangers names."""

    if metric in data_group:
        return metric

    if (
        metric == "granger"
        and "grangers"
        in data_group
    ):
        return "grangers"

    return None


def load_metric(
    recording: Recording,
    metric: str,
    reference_frequencies: np.ndarray,
) -> np.ndarray | None:
    """
    Load one spectral metric.

    Expected shapes:

        power:
            [time, frequency, region]

        coherence:
            [time, frequency, region, region]

        granger:
            [time, frequency, target, source]
    """

    with h5py.File(
        recording.path,
        "r",
    ) as h5_file:

        data_group = h5_file[
            "data"
        ]

        dataset_name = (
            metric_dataset_name(
                data_group,
                metric,
            )
        )

        if dataset_name is None:
            return None

        dataset = data_group[
            dataset_name
        ]

        expected_dimensions = (
            3
            if metric == "power"
            else 4
        )

        if (
            dataset.ndim
            != expected_dimensions
        ):
            raise ValueError(
                f"{recording.name} "
                f"{metric} should have "
                f"{expected_dimensions} dimensions; "
                f"found {dataset.shape}."
            )

        if (
            dataset.shape[1]
            != recording.frequencies.size
        ):
            raise ValueError(
                "Frequency does not appear "
                f"to be axis 1 in "
                f"{recording.name} {metric}."
            )

        frequency_indices = (
            np.flatnonzero(
                (
                    recording.frequencies
                    >= 0.0
                )
                & (
                    recording.frequencies
                    <= 100.0
                )
            )
        )

        if (
            frequency_indices.size
            == 0
        ):
            raise ValueError(
                f"No frequencies from 0 to 100 Hz "
                f"were found in {recording.name}."
            )

        frequency_start = int(
            frequency_indices[0]
        )

        frequency_stop = (
            int(
                frequency_indices[-1]
            )
            + 1
        )

        region_indices = [
            recording.region_to_index[
                region
            ]
            for region
            in ANALYSIS_REGIONS
        ]

        if min(
            region_indices
        ) < 0:
            raise IndexError(
                f"{recording.name} contains "
                "a negative region index."
            )

        if metric == "power":
            if (
                max(
                    region_indices
                )
                >= dataset.shape[2]
            ):
                raise IndexError(
                    "Region mapping exceeds "
                    f"{recording.name} "
                    "power-array dimensions."
                )

            metric_array = np.asarray(
                dataset[
                    :,
                    frequency_start:
                    frequency_stop,
                    :,
                ],
                dtype=float,
            )

            metric_array = metric_array[
                :,
                :,
                region_indices,
            ]

        else:
            if (
                max(
                    region_indices
                )
                >= dataset.shape[2]
                or max(
                    region_indices
                )
                >= dataset.shape[3]
            ):
                raise IndexError(
                    "Region mapping exceeds "
                    f"{recording.name} "
                    f"{metric}-array dimensions."
                )

            metric_array = np.asarray(
                dataset[
                    :,
                    frequency_start:
                    frequency_stop,
                    :,
                    :,
                ],
                dtype=float,
            )

            metric_array = metric_array[
                :,
                :,
                region_indices,
                :,
            ]

            metric_array = metric_array[
                :,
                :,
                :,
                region_indices,
            ]

    local_frequencies = (
        recording.frequencies[
            frequency_start:
            frequency_stop
        ]
    )

    if (
        local_frequencies.size
        != reference_frequencies.size
        or not np.allclose(
            local_frequencies,
            reference_frequencies,
        )
    ):
        metric_array = (
            nearest_resample_axis(
                metric_array,
                local_frequencies,
                reference_frequencies,
                axis=1,
            )
        )

    metric_array[
        ~np.isfinite(
            metric_array
        )
    ] = np.nan

    return metric_array


## Feature Extraction

Convert raw metric arrays into the permitted regional or connection-level feature rows used for testing.


In [ ]:
def extract_features(
    metric_array: np.ndarray,
    metric: str,
) -> tuple[
    np.ndarray,
    list[
        dict[
            str,
            str,
        ]
    ],
]:
    """
    Return an array shaped:

        [time, frequency, permitted feature]
    """

    region_index = {
        region: index
        for (
            index,
            region,
        ) in enumerate(
            ANALYSIS_REGIONS
        )
    }

    feature_arrays: list[
        np.ndarray
    ] = []

    feature_definitions: list[
        dict[
            str,
            str,
        ]
    ] = []

    if metric == "power":
        for region in POWER_FEATURES:
            feature_arrays.append(
                metric_array[
                    :,
                    :,
                    region_index[
                        region
                    ],
                ]
            )

            feature_definitions.append(
                {
                    "directionality": (
                        "regional"
                    ),
                    "label": region,
                    "matrix_target": (
                        region
                    ),
                    "matrix_source": "",
                }
            )

    elif metric == "coherence":
        for (
            display_label,
            first_region,
            second_region,
        ) in COHERENCE_FEATURES:

            forward = metric_array[
                :,
                :,
                region_index[
                    first_region
                ],
                region_index[
                    second_region
                ],
            ]

            reverse = metric_array[
                :,
                :,
                region_index[
                    second_region
                ],
                region_index[
                    first_region
                ],
            ]

            nondirectional_value = (
                np.nanmean(
                    np.stack(
                        (
                            forward,
                            reverse,
                        ),
                        axis=-1,
                    ),
                    axis=-1,
                )
            )

            feature_arrays.append(
                nondirectional_value
            )

            feature_definitions.append(
                {
                    "directionality": (
                        "nondirectional"
                    ),
                    "label": (
                        display_label
                    ),
                    "matrix_target": (
                        first_region
                    ),
                    "matrix_source": (
                        second_region
                    ),
                }
            )

    elif metric == "granger":
        for (
            display_label,
            target_region,
            source_region,
        ) in GRANGER_FEATURES:

            directed_value = (
                metric_array[
                    :,
                    :,
                    region_index[
                        target_region
                    ],
                    region_index[
                        source_region
                    ],
                ]
            )

            feature_arrays.append(
                directed_value
            )

            feature_definitions.append(
                {
                    "directionality": (
                        "directed source to target"
                    ),
                    "label": (
                        display_label
                    ),
                    "matrix_target": (
                        target_region
                    ),
                    "matrix_source": (
                        source_region
                    ),
                }
            )

    else:
        raise ValueError(
            f"Unsupported metric: {metric}"
        )

    return (
        np.stack(
            feature_arrays,
            axis=-1,
        ),
        feature_definitions,
    )


## Transformation And Normalization

Apply metric-specific transforms and robust within-recording normalization before event alignment.


In [ ]:
def robust_recording_zscore(
    values: np.ndarray,
) -> np.ndarray:
    """
    Normalize each frequency-feature combination
    over the complete recording:

        z = (value - median) / (1.4826 × MAD)

    Standard deviation is used when MAD is zero
    or invalid.
    """

    recording_median = np.nanmedian(
        values,
        axis=0,
    )

    median_absolute_deviation = (
        np.nanmedian(
            np.abs(
                values
                - recording_median[
                    None,
                    ...,
                ]
            ),
            axis=0,
        )
    )

    robust_scale = (
        1.4826
        * median_absolute_deviation
    )

    standard_deviation = np.nanstd(
        values,
        axis=0,
    )

    invalid_scale = (
        ~np.isfinite(
            robust_scale
        )
        | (
            robust_scale
            < EPSILON
        )
    )

    robust_scale = np.where(
        invalid_scale,
        standard_deviation,
        robust_scale,
    )

    invalid_scale = (
        ~np.isfinite(
            robust_scale
        )
        | (
            robust_scale
            < EPSILON
        )
    )

    robust_scale = np.where(
        invalid_scale,
        np.nan,
        robust_scale,
    )

    return (
        values
        - recording_median[
            None,
            ...,
        ]
    ) / robust_scale[
        None,
        ...,
    ]


def transform_and_normalize(
    raw_features: np.ndarray,
    metric: str,
) -> np.ndarray:
    """Transform and robustly normalize one recording."""

    if metric == "power":
        transformed = np.log10(
            np.maximum(
                raw_features,
                EPSILON,
            )
        )

    elif metric == "coherence":
        transformed = np.clip(
            raw_features,
            0.0,
            1.0,
        )

    elif metric == "granger":
        transformed = np.maximum(
            raw_features,
            0.0,
        )

        if (
            GRANGER_TRANSFORM
            == "log1p"
        ):
            transformed = np.log1p(
                transformed
            )

        elif (
            GRANGER_TRANSFORM
            == "log10"
        ):
            transformed = np.log10(
                transformed
                + EPSILON
            )

        elif (
            GRANGER_TRANSFORM
            == "none"
        ):
            pass

        else:
            raise ValueError(
                "GRANGER_TRANSFORM must be "
                "'log1p', 'log10', or 'none'."
            )

    else:
        raise ValueError(
            f"Unsupported metric: {metric}"
        )

    return robust_recording_zscore(
        transformed
    )


## Event Alignment

Extract event-locked windows for each recording and average events onto a shared canonical time axis.


In [ ]:
def recording_event_average(
    feature_values: np.ndarray,
    event_onsets: np.ndarray,
    recording: Recording,
) -> np.ndarray | None:
    """
    Extract complete -2-to-+2-second windows around
    each event and average all events within a recording.

    Output shape:

        [relative time, frequency, permitted feature]
    """

    relative_times = np.arange(
        -PRE_SECONDS,
        (
            POST_SECONDS
            + recording.time_step
            / 2.0
        ),
        recording.time_step,
    )

    stored_time_centers = (
        recording.window_duration
        / 2.0
        + np.arange(
            feature_values.shape[0]
        )
        * recording.time_step
    )

    event_windows: list[
        np.ndarray
    ] = []

    for event_onset in np.asarray(
        event_onsets,
        dtype=float,
    ):
        desired_centers = (
            float(
                event_onset
            )
            + relative_times
        )

        # Require a complete -2 through +2 second window.

        if (
            desired_centers[0]
            < stored_time_centers[0]
            or desired_centers[-1]
            > stored_time_centers[-1]
        ):
            continue

        right_indices = np.clip(
            np.searchsorted(
                stored_time_centers,
                desired_centers,
            ),
            0,
            (
                stored_time_centers.size
                - 1
            ),
        )

        left_indices = np.clip(
            right_indices - 1,
            0,
            (
                stored_time_centers.size
                - 1
            ),
        )

        choose_left = (
            np.abs(
                stored_time_centers[
                    left_indices
                ]
                - desired_centers
            )
            <= np.abs(
                stored_time_centers[
                    right_indices
                ]
                - desired_centers
            )
        )

        selected_indices = (
            np.where(
                choose_left,
                left_indices,
                right_indices,
            )
        )

        mismatch = np.abs(
            stored_time_centers[
                selected_indices
            ]
            - desired_centers
        )

        if np.any(
            mismatch
            > (
                recording.time_step
                / 2.0
                + 1e-9
            )
        ):
            continue

        event_windows.append(
            feature_values[
                selected_indices
            ]
        )

    if not event_windows:
        return None

    recording_average = np.nanmean(
        np.stack(
            event_windows,
            axis=0,
        ),
        axis=0,
    )

    target_times = canonical_times()

    if (
        relative_times.size
        != target_times.size
        or not np.allclose(
            relative_times,
            target_times,
        )
    ):
        recording_average = (
            nearest_resample_axis(
                recording_average,
                relative_times,
                target_times,
                axis=0,
            )
        )

    return recording_average


## Frequency-band Averaging

Average event-locked activity within predefined frequency bands and append recording-level rows for statistics.


In [ ]:
def frequency_band_mask(
    frequencies: np.ndarray,
    band_index: int,
    low_hz: float,
    high_hz: float,
) -> np.ndarray:
    """Return the frequency bins for one canonical band."""

    if (
        band_index
        == len(
            FREQUENCY_BANDS
        ) - 1
    ):
        return (
            (
                frequencies
                >= low_hz
            )
            & (
                frequencies
                <= high_hz
            )
        )

    return (
        (
            frequencies
            >= low_hz
        )
        & (
            frequencies
            < high_hz
        )
    )


def append_recording_rows(
    rows: list[
        dict[
            str,
            Any,
        ]
    ],
    recording: Recording,
    behavior: str,
    condition: str,
    metric: str,
    event_average: np.ndarray,
    feature_definitions: list[
        dict[
            str,
            str,
        ]
    ],
    frequencies: np.ndarray,
) -> None:
    """
    Convert one recording average into one value
    for each feature, band, and time point.
    """

    for (
        band_index,
        (
            band_name,
            low_hz,
            high_hz,
        ),
    ) in enumerate(
        FREQUENCY_BANDS
    ):

        mask = frequency_band_mask(
            frequencies,
            band_index,
            low_hz,
            high_hz,
        )

        if not np.any(
            mask
        ):
            raise ValueError(
                f"No frequency bins were "
                f"found for {band_name}."
            )

        band_values = np.nanmean(
            event_average[
                :,
                mask,
                :,
            ],
            axis=1,
        )

        for (
            feature_index,
            definition,
        ) in enumerate(
            feature_definitions
        ):

            for (
                time_index,
                time_s,
            ) in enumerate(
                canonical_times()
            ):
                rows.append(
                    {
                        "day": (
                            recording.day
                        ),
                        "behavior": (
                            behavior
                        ),
                        "condition": (
                            condition
                        ),
                        "condition_display": (
                            CONDITIONS[
                                condition
                            ]["display"]
                        ),
                        "recording": (
                            recording.name
                        ),
                        "animal_id": (
                            recording.animal_id
                        ),
                        "metric": (
                            metric
                        ),
                        "directionality": (
                            definition[
                                "directionality"
                            ]
                        ),
                        "region_or_connection": (
                            definition[
                                "label"
                            ]
                        ),
                        "matrix_target": (
                            definition[
                                "matrix_target"
                            ]
                        ),
                        "matrix_source": (
                            definition[
                                "matrix_source"
                            ]
                        ),
                        "time_slot": (
                            format_time_slot(
                                float(
                                    time_s
                                ),
                                behavior,
                            )
                        ),
                        "time_center_s": (
                            float(
                                time_s
                            )
                        ),
                        "frequency_band": (
                            band_name
                        ),
                        "frequency_low_hz": (
                            low_hz
                        ),
                        "frequency_high_hz": (
                            high_hz
                        ),
                        "value": float(
                            band_values[
                                time_index,
                                feature_index,
                            ]
                        ),
                    }
                )


## Build Recording-level Table

Construct the long-form table of recording-level observations across days, metrics, behaviors, conditions, features, bands, and time slots.


In [ ]:
def build_recording_table() -> pd.DataFrame:
    """
    Average events within each recording.

    Individual events are not treated as independent
    observations.
    """

    rows: list[
        dict[
            str,
            Any,
        ]
    ] = []

    for (
        day,
        folder,
    ) in COLLECTION_PATHS.items():

        print(
            f"\nLoading {day}: "
            f"{folder}"
        )

        (
            recordings,
            reference_frequencies,
        ) = load_collection(
            day,
            folder,
        )

        # Unweighted frequency-band means require
        # evenly spaced frequency bins.

        if (
            reference_frequencies.size
            > 2
        ):
            spacing = np.diff(
                reference_frequencies
            )

            if not np.allclose(
                spacing,
                spacing[0],
                rtol=1e-6,
                atol=1e-9,
            ):
                raise ValueError(
                    f"{day} uses a nonuniform "
                    "frequency grid. Unweighted "
                    "frequency-band means would "
                    "not be valid."
                )

        unknown_recordings = [
            recording.name
            for recording
            in recordings
            if (
                recording.role
                == "unknown"
            )
        ]

        if unknown_recordings:
            raise ValueError(
                "Could not assign animal roles "
                f"for {day}:\n"
                + "\n".join(
                    f"  - {name}"
                    for name
                    in unknown_recordings
                )
                + "\nCheck that the recording name "
                "or subject metadata contains the "
                "expected animal ID."
            )

        for (
            recording_number,
            recording,
        ) in enumerate(
            recordings,
            start=1,
        ):

            # Recipient recordings are not involved in
            # any requested comparison.

            if (
                recording.role
                == "recipient"
            ):
                print(
                    f"  [{recording_number:02d}/"
                    f"{len(recordings):02d}] "
                    "skip recipient "
                    f"{recording.name}"
                )

                continue

            for metric in METRICS:
                metric_array = load_metric(
                    recording,
                    metric,
                    reference_frequencies,
                )

                if metric_array is None:
                    print(
                        f"  missing {metric}: "
                        f"{recording.name}"
                    )

                    continue

                (
                    raw_features,
                    feature_definitions,
                ) = extract_features(
                    metric_array,
                    metric,
                )

                normalized_features = (
                    transform_and_normalize(
                        raw_features,
                        metric,
                    )
                )

                number_of_time_bins = (
                    normalized_features.shape[0]
                )

                retained_nose_pokes = (
                    filtered_nose_pokes(
                        recording,
                        number_of_time_bins,
                    )
                )

                for behavior in BEHAVIORS:
                    for (
                        condition,
                        condition_information,
                    ) in CONDITIONS.items():

                        if (
                            condition_information[
                                "role"
                            ]
                            != recording.role
                        ):
                            continue

                        event_name = find_event(
                            recording,
                            condition_information[
                                "choice"
                            ],
                            behavior,
                        )

                        if event_name is None:
                            continue

                        if (
                            behavior
                            == "nose poke"
                        ):
                            event_onsets = (
                                retained_nose_pokes.get(
                                    event_name,
                                    np.array(
                                        [],
                                        dtype=float,
                                    ),
                                )
                            )

                        else:
                            event_onsets = (
                                event_onsets_seconds(
                                    recording,
                                    event_name,
                                    number_of_time_bins,
                                )
                            )

                        event_average = (
                            recording_event_average(
                                normalized_features,
                                event_onsets,
                                recording,
                            )
                        )

                        if (
                            event_average
                            is None
                        ):
                            continue

                        append_recording_rows(
                            rows=rows,
                            recording=recording,
                            behavior=behavior,
                            condition=condition,
                            metric=metric,
                            event_average=(
                                event_average
                            ),
                            feature_definitions=(
                                feature_definitions
                            ),
                            frequencies=(
                                reference_frequencies
                            ),
                        )

                del metric_array
                del raw_features
                del normalized_features

            print(
                f"  [{recording_number:02d}/"
                f"{len(recordings):02d}] "
                f"processed {recording.name} "
                f"({recording.animal_id}, "
                f"{recording.role})"
            )

    if not rows:
        raise RuntimeError(
            "No event-aligned feature rows "
            "were created."
        )

    recording_table = pd.DataFrame(
        rows
    )

    unique_key = [
        "day",
        "behavior",
        "condition",
        "recording",
        "metric",
        "region_or_connection",
        "time_center_s",
        "frequency_band",
    ]

    duplicate_mask = (
        recording_table.duplicated(
            unique_key,
            keep=False,
        )
    )

    if duplicate_mask.any():
        duplicate_example = (
            recording_table.loc[
                duplicate_mask,
                unique_key,
            ].head(10)
        )

        raise RuntimeError(
            "Duplicate recording-level feature "
            "rows were created:\n"
            + duplicate_example.to_string(
                index=False
            )
        )

    return recording_table


## Aggregate Repeated Recordings Within Each Animal

Collapse repeated recordings within each animal/condition before group comparisons so animals, not recordings, drive the tests.


In [ ]:
def aggregate_within_animal(
    recording_table: pd.DataFrame,
) -> pd.DataFrame:
    """
    Average repeated recordings within each animal.

    Each animal therefore contributes at most one
    value to each t-test.
    """

    grouping_columns = [
        "day",
        "behavior",
        "condition",
        "condition_display",
        "animal_id",
        "metric",
        "directionality",
        "region_or_connection",
        "matrix_target",
        "matrix_source",
        "time_slot",
        "time_center_s",
        "frequency_band",
        "frequency_low_hz",
        "frequency_high_hz",
    ]

    animal_table = (
        recording_table.groupby(
            grouping_columns,
            dropna=False,
            sort=False,
            as_index=False,
        )
        .agg(
            value=(
                "value",
                "mean",
            ),
            n_recordings=(
                "recording",
                "nunique",
            ),
        )
    )

    unique_key = [
        "day",
        "behavior",
        "condition",
        "animal_id",
        "metric",
        "region_or_connection",
        "time_center_s",
        "frequency_band",
    ]

    if animal_table.duplicated(
        unique_key
    ).any():
        raise RuntimeError(
            "Animal aggregation did not "
            "produce unique animal-level rows."
        )

    return animal_table


## T-tests

Run the configured pairwise statistical comparisons and assemble effect sizes, p-values, and significance flags.


In [ ]:
TEST_GROUP_COLUMNS = [
    "day",
    "behavior",
    "metric",
    "directionality",
    "region_or_connection",
    "matrix_target",
    "matrix_source",
    "time_slot",
    "time_center_s",
    "frequency_band",
    "frequency_low_hz",
    "frequency_high_hz",
]


RESULT_COLUMNS = [
    "day",
    "behavior",
    "comparison",
    "group_1",
    "group_2",
    "metric",
    "directionality",
    "region_or_connection",
    "matrix_target",
    "matrix_source",
    "time_slot",
    "time_center_s",
    "frequency_band",
    "frequency_low_hz",
    "frequency_high_hz",
    "test_type",
    "n_group_1_animals",
    "n_group_2_animals",
    "n_paired_animals",
    "mean_group_1",
    "mean_group_2",
    "mean_difference_group1_minus_group2",
    "t_statistic",
    "p_value",
    "alpha",
    "significant_uncorrected",
    "multiple_comparison_correction",
    "value_scale",
]


def run_comparison(
    animal_table: pd.DataFrame,
    comparison: Comparison,
) -> pd.DataFrame:
    """Run every permitted feature, band, and time test."""

    subset = animal_table.loc[
        animal_table[
            "condition"
        ].isin(
            (
                comparison.group_1,
                comparison.group_2,
            )
        )
    ].copy()

    result_rows: list[
        dict[
            str,
            Any,
        ]
    ] = []

    grouped = subset.groupby(
        TEST_GROUP_COLUMNS,
        dropna=False,
        sort=False,
    )

    for (
        feature_key,
        feature_block,
    ) in grouped:

        metadata = dict(
            zip(
                TEST_GROUP_COLUMNS,
                feature_key,
            )
        )

        first_group = feature_block.loc[
            feature_block[
                "condition"
            ] == comparison.group_1
        ]

        second_group = feature_block.loc[
            feature_block[
                "condition"
            ] == comparison.group_2
        ]

        if comparison.paired:
            # Match the two conditions by animal ID.

            paired = first_group[
                [
                    "animal_id",
                    "value",
                ]
            ].merge(
                second_group[
                    [
                        "animal_id",
                        "value",
                    ]
                ],
                on="animal_id",
                how="inner",
                suffixes=(
                    "_1",
                    "_2",
                ),
                validate="one_to_one",
            )

            paired = paired.loc[
                np.isfinite(
                    paired[
                        "value_1"
                    ]
                )
                & np.isfinite(
                    paired[
                        "value_2"
                    ]
                )
            ]

            values_1 = paired[
                "value_1"
            ].to_numpy(
                dtype=float
            )

            values_2 = paired[
                "value_2"
            ].to_numpy(
                dtype=float
            )

            n_paired_animals = len(
                paired
            )

            n_group_1_animals = (
                n_paired_animals
            )

            n_group_2_animals = (
                n_paired_animals
            )

            test_type = (
                "paired two-sided "
                "t-test by animal"
            )

            if (
                n_paired_animals
                >= 2
            ):
                test_result = (
                    stats.ttest_rel(
                        values_1,
                        values_2,
                        nan_policy="omit",
                    )
                )

                t_statistic = float(
                    test_result.statistic
                )

                p_value = float(
                    test_result.pvalue
                )

            else:
                t_statistic = np.nan
                p_value = np.nan

        else:
            first_finite = (
                first_group.loc[
                    np.isfinite(
                        first_group[
                            "value"
                        ]
                    )
                ]
            )

            second_finite = (
                second_group.loc[
                    np.isfinite(
                        second_group[
                            "value"
                        ]
                    )
                ]
            )

            overlapping_animals = (
                set(
                    first_finite[
                        "animal_id"
                    ]
                )
                & set(
                    second_finite[
                        "animal_id"
                    ]
                )
            )

            if overlapping_animals:
                raise RuntimeError(
                    "Independent comparison "
                    f"{comparison.display!r} "
                    "contains the same animal "
                    "in both groups: "
                    f"{sorted(overlapping_animals)}"
                )

            values_1 = first_finite[
                "value"
            ].to_numpy(
                dtype=float
            )

            values_2 = second_finite[
                "value"
            ].to_numpy(
                dtype=float
            )

            n_group_1_animals = len(
                values_1
            )

            n_group_2_animals = len(
                values_2
            )

            n_paired_animals = np.nan

            test_type = (
                "Welch independent "
                "two-sided t-test by animal"
            )

            if (
                n_group_1_animals
                >= 2
                and n_group_2_animals
                >= 2
            ):
                test_result = (
                    stats.ttest_ind(
                        values_1,
                        values_2,
                        equal_var=False,
                        nan_policy="omit",
                    )
                )

                t_statistic = float(
                    test_result.statistic
                )

                p_value = float(
                    test_result.pvalue
                )

            else:
                t_statistic = np.nan
                p_value = np.nan

        mean_group_1 = (
            float(
                np.nanmean(
                    values_1
                )
            )
            if len(
                values_1
            )
            else np.nan
        )

        mean_group_2 = (
            float(
                np.nanmean(
                    values_2
                )
            )
            if len(
                values_2
            )
            else np.nan
        )

        significant = bool(
            np.isfinite(
                p_value
            )
            and p_value < ALPHA
        )

        result_rows.append(
            {
                **metadata,
                "comparison": (
                    comparison.display
                ),
                "group_1": (
                    CONDITIONS[
                        comparison.group_1
                    ]["display"]
                ),
                "group_2": (
                    CONDITIONS[
                        comparison.group_2
                    ]["display"]
                ),
                "test_type": (
                    test_type
                ),
                "n_group_1_animals": (
                    n_group_1_animals
                ),
                "n_group_2_animals": (
                    n_group_2_animals
                ),
                "n_paired_animals": (
                    n_paired_animals
                ),
                "mean_group_1": (
                    mean_group_1
                ),
                "mean_group_2": (
                    mean_group_2
                ),
                "mean_difference_group1_minus_group2": (
                    mean_group_1
                    - mean_group_2
                ),
                "t_statistic": (
                    t_statistic
                ),
                "p_value": (
                    p_value
                ),
                "alpha": ALPHA,
                "significant_uncorrected": (
                    significant
                ),
                "multiple_comparison_correction": (
                    "none"
                ),
                "value_scale": (
                    value_scale_description(
                        str(
                            metadata[
                                "metric"
                            ]
                        )
                    )
                ),
            }
        )

    return pd.DataFrame(
        result_rows
    )


## MITU Significance-of-interest Report

Create a focused report for a curated set of metric/behavior/feature/frequency/time results of interest.


In [ ]:
MATCH_COLUMNS = [
    "day",
    "behavior",
    "metric",
    "region_or_connection",
    "matrix_target",
    "matrix_source",
    "time_center_s",
    "frequency_band",
]


def result_lookup(
    frame: pd.DataFrame,
) -> dict[
    tuple[
        Any,
        ...,
    ],
    pd.Series,
]:
    """Build an exact feature, time, and band lookup."""

    return {
        tuple(
            row[
                column
            ]
            for column
            in MATCH_COLUMNS
        ): row
        for (
            _,
            row,
        ) in frame.iterrows()
    }


def format_p_value(
    value: float,
) -> str:
    """Format a p-value for the text report."""

    if not np.isfinite(
        value
    ):
        return "not testable"

    if value < 0.0001:
        return f"{value:.2e}"

    return f"{value:.4f}"


def higher_group_text(
    row: pd.Series,
) -> str:
    """Identify which condition has the larger mean."""

    difference = float(
        row[
            "mean_difference_group1_minus_group2"
        ]
    )

    if (
        not np.isfinite(
            difference
        )
        or abs(
            difference
        ) < 1e-12
    ):
        return (
            "with equal estimated means"
        )

    higher_group = (
        row[
            "group_1"
        ]
        if difference > 0
        else row[
            "group_2"
        ]
    )

    return (
        "with a higher mean normalized "
        f"value for {higher_group}"
    )


def build_mitu_report(
    all_results: dict[
        str,
        pd.DataFrame,
    ],
) -> str:
    """
    Retain a feature only when:

    1. selfish vs. coop is significant;
    2. selfish alone vs. coop alone is testable
       and not significant;
    3. selfish vs. selfish alone or
       coop vs. coop alone is significant;
    4. behavior is nose poke;
    5. time is -2.0 through 0.0 seconds.
    """

    baseline_lookup = result_lookup(
        all_results[
            "selfish_alone_vs_coop_alone"
        ]
    )

    selfish_support_lookup = (
        result_lookup(
            all_results[
                "selfish_vs_selfish_alone"
            ]
        )
    )

    coop_support_lookup = (
        result_lookup(
            all_results[
                "coop_vs_coop_alone"
            ]
        )
    )

    target_results = all_results[
        "selfish_vs_coop"
    ].copy()

    target_results = target_results.loc[
        target_results[
            "behavior"
        ].eq(
            "nose poke"
        )
        & target_results[
            "time_center_s"
        ].between(
            -2.0,
            0.0,
            inclusive="both",
        )
        & target_results[
            "significant_uncorrected"
        ]
    ]

    findings: dict[
        str,
        list[str],
    ] = {
        "day 1": [],
        "day 8": [],
    }

    target_results = (
        target_results.sort_values(
            [
                "day",
                "time_center_s",
                "metric",
                "region_or_connection",
                "frequency_band",
            ],
            kind="stable",
        )
    )

    for (
        _,
        result_row,
    ) in target_results.iterrows():

        lookup_key = tuple(
            result_row[
                column
            ]
            for column
            in MATCH_COLUMNS
        )

        baseline_result = (
            baseline_lookup.get(
                lookup_key
            )
        )

        selfish_support_result = (
            selfish_support_lookup.get(
                lookup_key
            )
        )

        coop_support_result = (
            coop_support_lookup.get(
                lookup_key
            )
        )

        # A missing or untestable baseline is not
        # treated as evidence of no baseline effect.

        baseline_is_testable_and_nonsignificant = (
            baseline_result is not None
            and np.isfinite(
                float(
                    baseline_result[
                        "p_value"
                    ]
                )
            )
            and float(
                baseline_result[
                    "p_value"
                ]
            ) >= ALPHA
        )

        selfish_support_is_significant = (
            selfish_support_result
            is not None
            and bool(
                selfish_support_result[
                    "significant_uncorrected"
                ]
            )
        )

        coop_support_is_significant = (
            coop_support_result
            is not None
            and bool(
                coop_support_result[
                    "significant_uncorrected"
                ]
            )
        )

        if not (
            baseline_is_testable_and_nonsignificant
        ):
            continue

        if not (
            selfish_support_is_significant
            or coop_support_is_significant
        ):
            continue

        supporting_comparisons: list[
            str
        ] = []

        if (
            selfish_support_is_significant
        ):
            supporting_comparisons.append(
                "selfish vs. selfish alone "
                "(p="
                + format_p_value(
                    float(
                        selfish_support_result[
                            "p_value"
                        ]
                    )
                )
                + ")"
            )

        if (
            coop_support_is_significant
        ):
            supporting_comparisons.append(
                "coop vs. coop alone "
                "(p="
                + format_p_value(
                    float(
                        coop_support_result[
                            "p_value"
                        ]
                    )
                )
                + ")"
            )

        time_value = float(
            result_row[
                "time_center_s"
            ]
        )

        if abs(
            time_value
        ) < 1e-12:
            time_phrase = (
                "at the nose poke "
                "(0.0 s)"
            )

        else:
            time_phrase = (
                f"{abs(time_value):.1f} s "
                "before the nose poke"
            )

        feature_phrase = (
            f"{result_row['frequency_band']}-band "
            f"{result_row['region_or_connection']} "
            f"{result_row['metric']}"
        )

        supporting_verb = (
            "was"
            if len(
                supporting_comparisons
            ) == 1
            else "were"
        )

        findings[
            str(
                result_row[
                    "day"
                ]
            )
        ].append(
            f"{time_phrase}, "
            f"{feature_phrase} significantly "
            "differed between selfish and "
            "cooperative choices "
            "(p="
            + format_p_value(
                float(
                    result_row[
                        "p_value"
                    ]
                )
            )
            + ", "
            + higher_group_text(
                result_row
            )
            + "); the matching selfish-alone "
            "vs. coop-alone comparison was not "
            "significant (p="
            + format_p_value(
                float(
                    baseline_result[
                        "p_value"
                    ]
                )
            )
            + "), while "
            + " and ".join(
                supporting_comparisons
            )
            + f" {supporting_verb} significant."
        )

    lines = [
        "Mitu significance of interest",
        "",
        (
            "Criteria: nose-poke features from "
            "-2.0 through 0.0 seconds; selfish "
            f"vs. coop uncorrected p < {ALPHA}; "
            "the exact matching selfish-alone "
            "vs. coop-alone comparison was "
            "testable and not significant; and "
            "at least one exact matching selfish "
            "vs. selfish-alone or coop vs. "
            "coop-alone comparison was "
            "significant. No FDR or other "
            "multiple-comparison correction "
            "was applied."
        ),
        "",
    ]

    for day in (
        "day 1",
        "day 8",
    ):
        lines.append(
            day.title()
            + ":"
        )

        if findings[
            day
        ]:
            lines.extend(
                f"- {sentence}"
                for sentence
                in findings[
                    day
                ]
            )

        else:
            lines.append(
                "- No result met all criteria."
            )

        lines.append("")

    return (
        "\n".join(
            lines
        ).rstrip()
        + "\n"
    )


## Internal Logic Checks

Run synthetic checks for role assignment, event filtering, feature orientation, and statistical-table assumptions.


In [ ]:
def run_internal_logic_checks() -> None:
    """
    Verify the feature lists, frequency boundaries,
    time axis, Granger orientation, coherence
    averaging, and comparison types.
    """

    assert len(
        POWER_FEATURES
    ) == 6

    assert len(
        COHERENCE_FEATURES
    ) == 4

    assert len(
        GRANGER_FEATURES
    ) == 8

    assert np.allclose(
        canonical_times(),
        np.array(
            [
                -2.0,
                -1.5,
                -1.0,
                -0.5,
                0.0,
                0.5,
                1.0,
                1.5,
                2.0,
            ]
        ),
    )

    # Confirm that adjacent bands touch without
    # gaps or overlapping intervals.

    for (
        current_band,
        following_band,
    ) in zip(
        FREQUENCY_BANDS[:-1],
        FREQUENCY_BANDS[1:],
    ):
        assert np.isclose(
            current_band[2],
            following_band[1],
        )

    # Confirm boundary frequencies enter exactly one band.

    boundary_frequencies = np.array(
        [
            0.5,
            3.999,
            4.0,
            7.999,
            8.0,
            11.999,
            12.0,
            29.999,
            30.0,
            100.0,
        ]
    )

    membership_count = np.zeros(
        boundary_frequencies.size,
        dtype=int,
    )

    for (
        band_index,
        (
            _,
            low_hz,
            high_hz,
        ),
    ) in enumerate(
        FREQUENCY_BANDS
    ):
        membership_count += (
            frequency_band_mask(
                boundary_frequencies,
                band_index,
                low_hz,
                high_hz,
            ).astype(
                int
            )
        )

    assert np.all(
        membership_count
        == 1
    )

    # Synthetic matrix test for Granger target/source
    # orientation and coherence symmetry.

    number_of_regions = len(
        ANALYSIS_REGIONS
    )

    synthetic_matrix = np.zeros(
        (
            1,
            1,
            number_of_regions,
            number_of_regions,
        ),
        dtype=float,
    )

    for target_index in range(
        number_of_regions
    ):
        for source_index in range(
            number_of_regions
        ):
            synthetic_matrix[
                0,
                0,
                target_index,
                source_index,
            ] = (
                100
                * target_index
                + source_index
            )

    (
        granger_features,
        granger_definitions,
    ) = extract_features(
        synthetic_matrix,
        "granger",
    )

    assert (
        granger_features.shape[-1]
        == 8
    )

    region_index = {
        region: index
        for (
            index,
            region,
        ) in enumerate(
            ANALYSIS_REGIONS
        )
    }

    for (
        feature_index,
        (
            _,
            target_region,
            source_region,
        ),
    ) in enumerate(
        GRANGER_FEATURES
    ):
        expected_value = (
            100
            * region_index[
                target_region
            ]
            + region_index[
                source_region
            ]
        )

        assert (
            granger_features[
                0,
                0,
                feature_index,
            ]
            == expected_value
        )

        assert (
            granger_definitions[
                feature_index
            ]["matrix_target"]
            == target_region
        )

        assert (
            granger_definitions[
                feature_index
            ]["matrix_source"]
            == source_region
        )

    (
        coherence_features,
        _,
    ) = extract_features(
        synthetic_matrix,
        "coherence",
    )

    for (
        feature_index,
        (
            _,
            first_region,
            second_region,
        ),
    ) in enumerate(
        COHERENCE_FEATURES
    ):
        expected_value = np.mean(
            [
                (
                    100
                    * region_index[
                        first_region
                    ]
                    + region_index[
                        second_region
                    ]
                ),
                (
                    100
                    * region_index[
                        second_region
                    ]
                    + region_index[
                        first_region
                    ]
                ),
            ]
        )

        assert np.isclose(
            coherence_features[
                0,
                0,
                feature_index,
            ],
            expected_value,
        )

    comparison_types = {
        comparison.key: (
            comparison.paired
        )
        for comparison
        in COMPARISONS
    }

    assert comparison_types == {
        (
            "selfish_alone_vs_"
            "coop_alone"
        ): True,
        (
            "selfish_vs_"
            "selfish_alone"
        ): False,
        (
            "coop_vs_"
            "coop_alone"
        ): False,
        "selfish_vs_coop": True,
    }

    assert (
        0.0
        < ALPHA
        < 1.0
    )

    print(
        "Internal logic checks passed."
    )


## Main Analysis

Execute the full bulk testing workflow: checks, table building, aggregation, t-tests, report creation, and file export.


In [ ]:
def main() -> None:
    """Run the full analysis and write the requested outputs."""

    run_internal_logic_checks()

    print(
        "=" * 88
    )

    print(
        "LFP BULK T-TESTS"
    )

    print(
        "=" * 88
    )

    print(
        "Day 1 collection: "
        f"{COLLECTION_PATHS['day 1']}"
    )

    print(
        "Day 8 collection: "
        f"{COLLECTION_PATHS['day 8']}"
    )

    print(
        f"Output folder: "
        f"{OUTPUT_DIR}"
    )

    print(
        f"Uncorrected alpha: "
        f"{ALPHA}"
    )

    print(
        "Statistical unit: animal"
    )

    recording_table = (
        build_recording_table()
    )

    animal_table = (
        aggregate_within_animal(
            recording_table
        )
    )

    print(
        f"\nCreated "
        f"{len(recording_table):,} "
        "recording-level rows and "
        f"{len(animal_table):,} "
        "animal-level rows."
    )

    all_results = {
        comparison.key: (
            run_comparison(
                animal_table,
                comparison,
            )
        )
        for comparison
        in COMPARISONS
    }

    for comparison in COMPARISONS:
        complete_results = (
            all_results[
                comparison.key
            ]
        )

        significant_results = (
            complete_results.loc[
                complete_results[
                    "significant_uncorrected"
                ]
            ].copy()
        )

        if significant_results.empty:
            significant_results = (
                pd.DataFrame(
                    columns=(
                        RESULT_COLUMNS
                    )
                )
            )

        else:
            significant_results = (
                significant_results[
                    RESULT_COLUMNS
                ].sort_values(
                    [
                        "day",
                        "behavior",
                        "metric",
                        "region_or_connection",
                        "time_center_s",
                        "frequency_band",
                        "p_value",
                    ],
                    kind="stable",
                )
            )

        output_path = (
            OUTPUT_DIR
            / comparison.filename
        )

        significant_results.to_csv(
            output_path,
            index=False,
            float_format="%.12g",
        )

        testable_count = int(
            complete_results[
                "p_value"
            ].notna().sum()
        )

        print(
            f"Wrote "
            f"{output_path.name}: "
            f"{len(significant_results)} "
            "significant rows from "
            f"{testable_count} "
            "testable comparisons."
        )

    report_path = (
        OUTPUT_DIR
        / (
            "mitu_significance_"
            "of_interest.txt"
        )
    )

    report_text = (
        build_mitu_report(
            all_results
        )
    )

    report_path.write_text(
        report_text,
        encoding="utf-8",
    )

    print(
        f"Wrote "
        f"{report_path.name}"
    )

    print(
        "\nDone."
    )


main()
